In [1]:
import sys,os,re,argparse,glob
import pandas as pd
from scipy.io import loadmat
from pathlib import Path
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

import numpy as np


In [2]:
ROOT = Path('/home/tntiniak/Work/observatory_benchmark')
PHYSICELL_DIR = ROOT / 'PhysiCell' / 'results' / 'mechanics_pushing_extended_hertz'
OUTPUT_FILENAME = 'physicell_mechanics_pushing_extended_hertz.csv'


In [3]:
records = []
for xml_path in sorted(PHYSICELL_DIR.glob('output*.xml')):
    root = ET.parse(xml_path).getroot()
    current_time = float(root.findtext('.//current_time'))
    mat_name = root.findtext('.//simplified_data/filename')
    if mat_name is None:
        raise ValueError(f'No cell data filename found in {xml_path}')

    cells = loadmat(PHYSICELL_DIR / mat_name)['cells']
    positions = cells[1:4, :].T
    radii = cells[37, :]
    if positions.shape[0] != 2:
        raise ValueError(f'Expected 2 cells, found {positions.shape[0]} in {mat_name}')

    distance = float(np.linalg.norm(positions[0] - positions[1]))
    overlap = float(radii[0] + radii[1] - distance)
    records.append({
        'time_min': current_time,
        'distance_um': distance,
        'radius_sum_um': float(radii[0] + radii[1]),
        'overlap_um': overlap,
    })

frame = pd.DataFrame.from_records(records).sort_values('time_min').reset_index(drop=True)
frame.to_csv(PHYSICELL_DIR / OUTPUT_FILENAME, index=False)